# Text summarizing with ChaptGPT
In this lesson, you will summarize text with a focus on specific topics.

## Setup

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [29]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)


def get_completion(prompt, model="gpt-3.5-turbo"): # Andrew mentioned that the prompt/ completion paradigm is preferable for this class
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
        max_tokens=100
    )
    return response.choices[0].message.content


## Text to summarize

In [5]:
prod_review = """
Got this panda plush toy for my daughter's birthday, \
who loves it and takes it everywhere. It's soft and \
super cute, and its face has a friendly look. It's \
a bit small for what I paid though. I think there \
might be other options that are bigger for the \
same price. It arrived a day earlier than expected, \
so I got to play with it myself before I gave it \
to her.
"""

In [11]:
prod_review

"\nGot this panda plush toy for my daughter's birthday, who loves it and takes it everywhere. It's soft and super cute, and its face has a friendly look. It's a bit small for what I paid though. I think there might be other options that are bigger for the same price. It arrived a day earlier than expected, so I got to play with it myself before I gave it to her.\n"

## Summarize with a word/sentence/character limit

In [12]:
prompt = f"""
Your task is to generate a short summary of a product \
review from an ecommerce site. 

Summarize the review below, delimited by triple 
backticks, in at most 30 words. 

Review: ```{prod_review}```
"""

response = get_completion(prompt)
print(response)


Summary: 
Cute panda plush toy loved by daughter, soft and friendly-looking. Small for price, but arrived early. Consider larger options for same cost.


## Summarize with a focus on shipping and delivery

In [13]:
prompt = f"""
Your task is to generate a short summary of a product \
review from an ecommerce site to give feedback to the \
Shipping deparmtment. 

Summarize the review below, delimited by triple 
backticks, in at most 30 words, and focusing on any aspects \
that mention shipping and delivery of the product. 

Review: ```{prod_review}```
"""

response = get_completion(prompt)
print(response)


The customer was pleased with the early delivery of the panda plush toy, but felt it was slightly small for the price paid.


## Summarize with a focus on price and value

In [14]:
prompt = f"""
Your task is to generate a short summary of a product \
review from an ecommerce site to give feedback to the \
pricing deparmtment, responsible for determining the \
price of the product.  

Summarize the review below, delimited by triple 
backticks, in at most 30 words, and focusing on any aspects \
that are relevant to the price and perceived value. 

Review: ```{prod_review}```
"""

response = get_completion(prompt)
print(response)


The panda plush toy is soft, cute, and loved by the daughter, but the reviewer feels it's a bit small for the price paid. Arrived early.


#### Comment
- Summaries include topics that are not related to the topic of focus.

## Try "extract" instead of "summarize"

In [6]:
prompt = f"""
Your task is to extract relevant information from \
a product review from an ecommerce site to give \
feedback to the Shipping department. \
\
From the review below, delimited by triple quotes \
extract the information relevant to shipping and \
delivery. Limit to 30 words. \
\
Review: ```{prod_review}```
"""

response = get_completion(prompt)
print(response)

Feedback: The product arrived a day earlier than expected, which was a pleasant surprise. Consider offering larger options for the same price to improve customer satisfaction.


## Summarize multiple product reviews

In [7]:
review_1 = prod_review 
# review for a standing lamp
review_2 = """
Needed a nice lamp for my bedroom, and this one \
had additional storage and not too high of a price \
point. Got it fast - arrived in 2 days. The string \
to the lamp broke during the transit and the company \
happily sent over a new one. Came within a few days \
as well. It was easy to put together. Then I had a \
missing part, so I contacted their support and they \
very quickly got me the missing piece! Seems to me \
to be a great company that cares about their customers \
and products. 
"""
# review for an electric toothbrush
review_3 = """
My dental hygienist recommended an electric toothbrush, \
which is why I got this. The battery life seems to be \
pretty impressive so far. After initial charging and \
leaving the charger plugged in for the first week to \
condition the battery, I've unplugged the charger and \
been using it for twice daily brushing for the last \
3 weeks all on the same charge. But the toothbrush head \
is too small. I've seen baby toothbrushes bigger than \
this one. I wish the head was bigger with different \
length bristles to get between teeth better because \
this one doesn't.  Overall if you can get this one \
around the $50 mark, it's a good deal. The manufactuer's \
replacements heads are pretty expensive, but you can \
get generic ones that're more reasonably priced. This \
toothbrush makes me feel like I've been to the dentist \
every day. My teeth feel sparkly clean! 
"""
# review for a blender
review_4 = """
So, they still had the 17 piece system on seasonal \
sale for around $49 in the month of November, about \
half off, but for some reason (call it price gouging) \
around the second week of December the prices all went \
up to about anywhere from between $70-$89 for the same \
system. And the 11 piece system went up around $10 or \
so in price also from the earlier sale price of $29. \
So it looks okay, but if you look at the base, the part \
where the blade locks into place doesn't look as good \
as in previous editions from a few years ago, but I \
plan to be very gentle with it (example, I crush \
very hard items like beans, ice, rice, etc. in the \
blender first then pulverize them in the serving size \
I want in the blender then switch to the whipping \
blade for a finer flour, and use the cross cutting blade \
first when making smoothies, then use the flat blade \
if I need them finer/less pulpy). Special tip when making \
smoothies, finely cut and freeze the fruits and \
vegetables (if using spinach-lightly stew soften the \
spinach then freeze until ready for use-and if making \
sorbet, use a small to medium sized food processor) \
that you plan to use that way you can avoid adding so \
much ice if at all-when making your smoothie. \
After about a year, the motor was making a funny noise. \
I called customer service but the warranty expired \
already, so I had to buy another one. FYI: The overall \
quality has gone done in these types of products, so \
they are kind of counting on brand recognition and \
consumer loyalty to maintain sales. Got it in about \
two days.
"""
reviews = [review_1, review_2, review_3, review_4]

In [8]:
for i in range(len(reviews)):
    prompt = f"""
    Your task is to generate a short summary of a product \
    review from an ecommerce site.\
    \
    Summarize the review below, delimited by triple \
    backticks in at most 20 words.\
    \
    Review: ```{reviews[i]}```
    """

    response = get_completion(prompt)
    print(i, response, "\n")

0 Summary: 
Adorable panda plush toy loved by daughter, soft and cute, but smaller than expected for the price. Arrived early. 

1 ```
Affordable lamp with storage, fast delivery, excellent customer service for missing parts. Great company that cares.
``` 

2 Impressive battery life, small head, good deal for $50, generic replacement heads recommended, leaves teeth feeling clean. 

3 17 piece system on sale for $49 in November, price increased to $70-$89 in December. Quality decline, motor issues after a year. 



# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [10]:
import pandas as pd

import os

In [11]:
os.listdir()

['.env',
 '.git',
 '.ipynb_checkpoints',
 'lab-text-summarizing-with-chatgpt.ipynb',
 'README.md',
 'TestReviews.csv']

In [13]:
reviews = pd.read_csv('TestReviews.csv')

In [30]:
review_list = reviews.dropna()['review'].sample(5).to_list()

In [35]:
# reviews

In [34]:
for index, review in enumerate(review_list) :
    prompt = f"""
    Your task is to generate a concise review summary of the product or service discussed \
    in the review below, delimited by triple backticks.

    Start your response in the following format:

    Coffeeshop (example of a reviewed place): Review summary (short and to the point).

    Capture the key points and important aspects for the business manager \
    and owner to assess and further improve the business.

    Review: ```{review}```
    """

    response = get_completion(prompt)
    print(f"Review {index + 1}:\n{review}\n\nSummary:\n{response}\n")

Review 1:
Saltiest (and most expensive) noodles ever!For a whopping $13, I got the equivalent of a bowl of noodles that would've cost $4 at a normal noodle shop.  I asked for udon noodles with my Cantonese Dumpling noodles.  Instead, I got egg noodles.  Which was fine, since egg noodles were probably the better choice for a noodle soup with dumplings.  However, I must've downed a whole pitcher of water by the time I was through with the noodles. So, one star for the price, messing up my order, and the saltiness that was pervasive in all of their noodle soups.


Summary:
Noodle Shop: Overpriced and excessively salty noodles, incorrect order, disappointing experience.

Review 2:
This guy altered a pair of my jeans and a fancy dress of mine, and both items came out PERFECT!Quick turn around time too.Love it.


Summary:
Tailoring Service: Excellent alterations with quick turnaround time. Highly satisfied with the results.

Review 3:
Awesome little home decor shop in Pittsburg.  We stumbled

In [36]:
# job postings

In [37]:
os.listdir()

['.env',
 '.git',
 '.ipynb_checkpoints',
 'data job posts.csv',
 'lab-text-summarizing-with-chatgpt.ipynb',
 'README.md',
 'TestReviews.csv']

In [38]:
jobs = pd.read_csv('data job posts.csv')

In [40]:
len(jobs)

19001

In [39]:
jobs.isna().sum()

jobpost                 0
date                    0
Title                  28
Company                 7
AnnouncementCode    17793
Term                11325
Eligibility         14071
Audience            18361
StartDate            9326
Duration             8203
Location               32
JobDescription       3892
JobRequirment        2522
RequiredQual          484
Salary               9379
ApplicationP           60
OpeningDate           706
Deadline               65
Notes               16790
AboutC               6531
Attach              17442
Year                    0
Month                   0
IT                      0
dtype: int64

In [41]:
jobs.columns

Index(['jobpost', 'date', 'Title', 'Company', 'AnnouncementCode', 'Term',
       'Eligibility', 'Audience', 'StartDate', 'Duration', 'Location',
       'JobDescription', 'JobRequirment', 'RequiredQual', 'Salary',
       'ApplicationP', 'OpeningDate', 'Deadline', 'Notes', 'AboutC', 'Attach',
       'Year', 'Month', 'IT'],
      dtype='str')

In [44]:
jobs_clean = jobs[['Title', 'Company', 'Salary', 'JobDescription']].dropna()

In [46]:
jobs_clean_sample = jobs_clean.sample(5)

jobs_clean_sample

,Title,Company,Salary,JobDescription
11786,Java Web Developer,SFL LLC,"Highly competitive, commensurate with experien...",SFL is seeking for talented and motivated deve...
5138,Copywriter,Sharm Holding LLC,Negotiable,"""Sharm Holding"" LLC is looking for a highly mo..."
13160,Lead Technical Writer,OMD LLC,Highly competitive,OMD LLC is looking for a Technical Writer to\r...
13335,Risk Manager,BTA Bank CJSC,"Negotiable, depending on the previous experien...",The Risk Manager will be part of BTA Bank CJSC...
479,Chief Accountant,ACRA Credit Bureau,Competitive wages based on previous experience...,The incumbent will start as the bearer of the ...


In [50]:
for index in range(5):
    prompt = f"""
    Your task is to generate a concise summary of the job listing. \
    

    Look at the Title, Company, Salary & JobDescription componetns.

    Capture the key points and important aspects for the potential applicant to know. \
    Keep it short and to the point

    Title: ```{jobs_clean_sample.iloc[index, 0]}```
    Company: ```{jobs_clean_sample.iloc[index, 1]}```
    Salary: ```{jobs_clean_sample.iloc[index, 2]}```
    JobDescription: ```{jobs_clean_sample.iloc[index, 3]}```
    """

    response = get_completion(prompt)
    print(f"""
        Title: {jobs_clean_sample.iloc[index, 0]}
        Company: {jobs_clean_sample.iloc[index, 1]}
        Salary: {jobs_clean_sample.iloc[index, 2]}
        
        
        {response}
        
    """)


        Title: Java Web Developer
        Company: SFL LLC
        Salary: Highly competitive, commensurate with experience.
Medical insurance, bonus program and personal training plan.


        Summary: SFL LLC is hiring a Java Web Developer to join their team. The salary is highly competitive and includes benefits such as medical insurance, bonus program, and personal training plan. Candidates should be talented and motivated developers.

    

        Title: Copywriter
        Company: Sharm Holding LLC
        Salary: Negotiable


        Summary: Sharm Holding LLC is seeking a motivated Copywriter to manage the Creative Department's workflow. Salary is negotiable.

    

        Title: Lead Technical Writer
        Company: OMD LLC
        Salary: Highly competitive


        Summary:
OMD LLC is seeking a Lead Technical Writer to participate in product documentation design and development. Highly competitive salary offered.

    

        Title: Risk Manager
        Company: BTA

In [52]:
"""
- summaries seem to work fine as long as necessary info is provided 
- formmat changes if not specified

"""

'\n- summaries seem to work fine as long as necessary info is provided \n- formmat changes if not specified\n\n'